In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [3]:
%pip install geopandas scipy

Note: you may need to restart the kernel to use updated packages.


In [4]:
import geopandas as gpd
from scipy import stats

print("GeoPandas version:", gpd.__version__)
print("SciPy version:", stats.__version__ if hasattr(stats, "__version__") else "Installed")

GeoPandas version: 1.2.0
SciPy version: Installed


In [5]:
df = pd.read_csv(
    r"../data/processed/nfhs5_india_district_child_malnutrition_factual.csv"
)

metadata = pd.read_csv(
    r"../data/processed/dataset_metadata_dictionary.csv"
)

geo = gpd.read_file(
    r"../data/geo/india_district_boundary.geojson"
)

In [6]:
print("MAIN DATASET")
print("Shape:", df.shape)
print("Districts:", df["district_name"].nunique())
print("States:", df["state_name"].nunique())

print("\nMETADATA")
print("Shape:", metadata.shape)

print("\nGEOJSON")
print("Shape:", geo.shape)
print("CRS:", geo.crs)

MAIN DATASET
Shape: (706, 14)
Districts: 699
States: 37

METADATA
Shape: (14, 8)

GEOJSON
Shape: (594, 12)
CRS: EPSG:4326


In [7]:
display(df.dtypes)

district_name                   str
state_name                      str
district_code               float64
state_code                    int64
stunting_pct                float64
wasting_pct                 float64
underweight_pct             float64
female_literacy_pct         float64
improved_sanitation_pct     float64
improved_water_pct          float64
anc_4plus_pct               float64
child_anemia_pct            float64
early_marriage_pct          float64
institutional_births_pct    float64
dtype: object

In [8]:
missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    )
})

display(missing)

,Missing_Count,Missing_Percentage
district_name,0,0.000000
state_name,0,0.000000
district_code,65,9.206799
state_code,0,0.000000
stunting_pct,0,0.000000
wasting_pct,0,0.000000
underweight_pct,0,0.000000
female_literacy_pct,0,0.000000
improved_sanitation_pct,0,0.000000
improved_water_pct,0,0.000000


In [9]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
district_code,641.0,15.800312,13.729100,1.0,6.000,12.00,22.000,71.0
state_code,706.0,17.781870,9.829497,1.0,9.000,18.50,24.000,36.0
stunting_pct,706.0,33.511190,8.471402,13.2,27.300,32.80,39.200,60.6
wasting_pct,706.0,18.538102,6.482302,4.5,13.900,18.10,22.775,48.0
underweight_pct,706.0,29.501841,9.681756,7.2,22.100,29.35,36.200,62.4
female_literacy_pct,706.0,73.087394,12.722809,37.1,64.450,73.80,82.900,99.7
improved_sanitation_pct,706.0,71.945751,14.298168,29.2,62.025,73.75,83.300,99.9
improved_water_pct,706.0,93.779603,8.663805,41.2,92.100,97.00,99.300,100.0
anc_4plus_pct,706.0,60.465156,20.298684,4.4,45.000,62.40,76.075,98.7
child_anemia_pct,706.0,65.812890,12.068456,24.9,59.500,67.70,73.975,95.5


In [10]:
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate district-state combinations:",
    df.duplicated(
        subset=["district_name", "state_name"]
    ).sum()
)

Duplicate rows: 0
Duplicate district-state combinations: 0


In [11]:
percentage_columns = [
    col for col in df.columns
    if col.endswith("_pct")
]

for col in percentage_columns:

    invalid = df[
        (df[col] < 0) | (df[col] > 100)
    ]

    print(
        col,
        "Invalid values:",
        len(invalid)
    )

stunting_pct Invalid values: 0
wasting_pct Invalid values: 0
underweight_pct Invalid values: 0
female_literacy_pct Invalid values: 0
improved_sanitation_pct Invalid values: 0
improved_water_pct Invalid values: 0
anc_4plus_pct Invalid values: 0
child_anemia_pct Invalid values: 0
early_marriage_pct Invalid values: 0
institutional_births_pct Invalid values: 0


In [12]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [13]:
print("Main Dataset:", df.shape)
print("Metadata:", metadata.shape)
print("Geographical Dataset:", geo.shape)

Main Dataset: (706, 14)
Metadata: (14, 8)
Geographical Dataset: (594, 12)


In [14]:
display(df.head())
display(metadata.head())
display(geo.head())

,district_name,state_name,district_code,state_code,stunting_pct,wasting_pct,underweight_pct,female_literacy_pct,improved_sanitation_pct,improved_water_pct,anc_4plus_pct,child_anemia_pct,early_marriage_pct,institutional_births_pct
0,Nicobar,Andaman & Nicobar Island,1.0,35,21.6,15.7,24.6,87.5,83.5,98.8,71.7,37.7,11.4,97.8
1,North & Middle Andaman,Andaman & Nicobar Island,2.0,35,27.0,27.0,42.8,84.0,86.4,92.2,79.2,30.4,15.4,97.7
2,South Andaman,Andaman & Nicobar Island,3.0,35,21.1,12.6,17.4,86.7,89.3,97.9,85.9,43.4,17.1,99.5
3,Anantapur,Andhra Pradesh,12.0,28,36.0,19.3,40.6,63.6,71.3,98.8,66.6,55.8,37.3,94.7
4,Chittoor,Andhra Pradesh,13.0,28,27.1,14.8,27.9,69.3,74.6,98.5,65.3,54.9,28.1,97.1


,variable_name,indicator_category,operational_definition,data_type,unit,nfhs5_indicator_name,nfhs5_indicator_number,un_sdg_target
0,district_name,Primary Key / Geographic Identifier,Official administrative name of the district a...,string,District Name,District,NaN,Administrative Geo-Reference
1,state_name,Primary Key / Geographic Hierarchy,Official administrative name of the Indian Sta...,string,State / UT Name,State,NaN,Administrative Geo-Reference
2,district_code,Geospatial Linkage Key,Census of India 2011 district code assigned to...,integer,Census ID,DT_CEN_CD,NaN,Geospatial Boundary Alignment
3,state_code,Geospatial Linkage Key,Census of India 2011 state code assigned to th...,integer,Census ID,ST_CEN_CD,NaN,Geospatial Boundary Alignment
4,stunting_pct,Anthropometric Dependent Variable (Target 2.2),Percentage of children under age 5 years whose...,float,%,Children under 5 years who are stunted (height...,73.0,SDG 2.2.1 (Prevalence of stunting among childr...


,ID_0,ISO,NAME_0,ID_1,NAME_1,ID_2,NAME_2,NL_NAME_2,VARNAME_2,TYPE_2,ENGTYPE_2,geometry
0,105,IND,India,1,Andaman and Nicobar,1,Andaman Islands,None,NaN,District,District,"MULTIPOLYGON (((92.51583 10.89764, 92.51611 10..."
1,105,IND,India,1,Andaman and Nicobar,2,Nicobar Islands,None,NaN,District,District,"MULTIPOLYGON (((93.78773 6.85264, 93.78849 6.8..."
2,105,IND,India,2,Andhra Pradesh,3,Adilabad,None,NaN,District,District,"POLYGON ((78.33625 19.88319, 78.34669 19.8814,..."
3,105,IND,India,2,Andhra Pradesh,4,Anantapur,None,"Anantpur, Ananthapur",District,District,"POLYGON ((77.69 15.17628, 77.69378 15.17347, 7..."
4,105,IND,India,2,Andhra Pradesh,5,Chittoor,None,Chitoor|Chittor,District,District,"POLYGON ((78.47611 13.9368, 78.48208 13.93007,..."


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 706 entries, 0 to 705
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   district_name             706 non-null    str    
 1   state_name                706 non-null    str    
 2   district_code             641 non-null    float64
 3   state_code                706 non-null    int64  
 4   stunting_pct              706 non-null    float64
 5   wasting_pct               706 non-null    float64
 6   underweight_pct           706 non-null    float64
 7   female_literacy_pct       706 non-null    float64
 8   improved_sanitation_pct   706 non-null    float64
 9   improved_water_pct        706 non-null    float64
 10  anc_4plus_pct             706 non-null    float64
 11  child_anemia_pct          706 non-null    float64
 12  early_marriage_pct        706 non-null    float64
 13  institutional_births_pct  706 non-null    float64
dtypes: float64(11), int64

In [16]:
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   variable_name           14 non-null     str    
 1   indicator_category      14 non-null     str    
 2   operational_definition  14 non-null     str    
 3   data_type               14 non-null     str    
 4   unit                    14 non-null     str    
 5   nfhs5_indicator_name    14 non-null     str    
 6   nfhs5_indicator_number  10 non-null     float64
 7   un_sdg_target           14 non-null     str    
dtypes: float64(1), str(7)
memory usage: 4.9 KB


In [17]:
geo.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 594 entries, 0 to 593
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   ID_0       594 non-null    int32   
 1   ISO        594 non-null    str     
 2   NAME_0     594 non-null    str     
 3   ID_1       594 non-null    int32   
 4   NAME_1     594 non-null    str     
 5   ID_2       594 non-null    int32   
 6   NAME_2     594 non-null    str     
 7   NL_NAME_2  0 non-null      object  
 8   VARNAME_2  177 non-null    str     
 9   TYPE_2     594 non-null    str     
 10  ENGTYPE_2  594 non-null    str     
 11  geometry   594 non-null    geometry
dtypes: geometry(1), int32(3), object(1), str(7)
memory usage: 75.8+ KB


In [18]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
district_code,641.0,15.800312,13.729100,1.0,6.000,12.00,22.000,71.0
state_code,706.0,17.781870,9.829497,1.0,9.000,18.50,24.000,36.0
stunting_pct,706.0,33.511190,8.471402,13.2,27.300,32.80,39.200,60.6
wasting_pct,706.0,18.538102,6.482302,4.5,13.900,18.10,22.775,48.0
underweight_pct,706.0,29.501841,9.681756,7.2,22.100,29.35,36.200,62.4
female_literacy_pct,706.0,73.087394,12.722809,37.1,64.450,73.80,82.900,99.7
improved_sanitation_pct,706.0,71.945751,14.298168,29.2,62.025,73.75,83.300,99.9
improved_water_pct,706.0,93.779603,8.663805,41.2,92.100,97.00,99.300,100.0
anc_4plus_pct,706.0,60.465156,20.298684,4.4,45.000,62.40,76.075,98.7
child_anemia_pct,706.0,65.812890,12.068456,24.9,59.500,67.70,73.975,95.5


In [19]:
missing = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': df.isnull().mean() * 100
})

display(missing)

,Missing_Count,Missing_Percentage
district_name,0,0.000000
state_name,0,0.000000
district_code,65,9.206799
state_code,0,0.000000
stunting_pct,0,0.000000
wasting_pct,0,0.000000
underweight_pct,0,0.000000
female_literacy_pct,0,0.000000
improved_sanitation_pct,0,0.000000
improved_water_pct,0,0.000000


In [20]:
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate district-state combinations:",
    df.duplicated(
        subset=['district_name', 'state_name']
    ).sum()
)

Exact duplicate rows: 0
Duplicate district-state combinations: 0


In [21]:
print("Geometry types:")
print(geo.geom_type.value_counts())

print("Invalid geometries:", (~geo.geometry.is_valid).sum())

print("Empty geometries:", geo.geometry.is_empty.sum())

Geometry types:
Polygon         518
MultiPolygon     76
Name: count, dtype: int64
Invalid geometries: 0
Empty geometries: 0


In [22]:
percentage_cols = [
    col for col in df.columns
    if col.endswith('_pct')
]

for col in percentage_cols:
    print(
        col,
        "Minimum:", df[col].min(),
        "Maximum:", df[col].max()
    )

stunting_pct Minimum: 13.2 Maximum: 60.6
wasting_pct Minimum: 4.5 Maximum: 48.0
underweight_pct Minimum: 7.2 Maximum: 62.4
female_literacy_pct Minimum: 37.1 Maximum: 99.7
improved_sanitation_pct Minimum: 29.2 Maximum: 99.9
improved_water_pct Minimum: 41.2 Maximum: 100.0
anc_4plus_pct Minimum: 4.4 Maximum: 98.7
child_anemia_pct Minimum: 24.9 Maximum: 95.5
early_marriage_pct Minimum: 0.0 Maximum: 80.9
institutional_births_pct Minimum: 21.4 Maximum: 100.0


In [23]:
print("Exact duplicate rows:", df.duplicated().sum())

print("Duplicate district-state pairs:",
      df.duplicated(
          subset=['district_name', 'state_name']
      ).sum())

print("Duplicate district codes:",
      df.duplicated(
          subset=['state_code', 'district_code'],
          keep=False
      ).sum())

Exact duplicate rows: 0
Duplicate district-state pairs: 0
Duplicate district codes: 70


In [24]:
missing_codes = df[df['district_code'].isnull()]

print("Missing district codes:", len(missing_codes))

display(
    missing_codes.groupby('state_name')
    .size()
    .sort_values(ascending=False)
    .to_frame('Missing_District_Codes')
)

Missing district codes: 65


,Missing_District_Codes
state_name,
Telangana,19
Chhattisgarh,9
Gujarat,7
Assam,6
Meghalaya,5
Tripura,4
Arunachal Pradesh,4
Uttar Pradesh,4
NCT of Delhi,2


In [25]:
print("Number of states in CSV:", df['state_name'].nunique())

print("\nDistricts per state:")

display(
    df.groupby('state_name')['district_name']
    .nunique()
    .sort_values(ascending=False)
)

Number of states in CSV: 37

Districts per state:


state_name
Uttar Pradesh               75
Madhya Pradesh              51
Bihar                       38
Maharashtra                 36
Assam                       33
Rajasthan                   33
Gujarat                     33
Tamil Nadu                  32
Telangana                   31
Odisha                      30
Karnataka                   30
Chhattisgarh                27
Jharkhand                   23
Haryana                     22
Punjab                      22
Arunachal Pradesh           20
West Bengal                 20
Jammu & Kashmir             20
Kerala                      14
Uttarakhand                 13
Andhra Pradesh              13
Himachal Pradesh            12
Nagaland                    11
NCT of Delhi                11
Meghalaya                   11
Manipur                      9
Tripura                      8
Mizoram                      8
Puducherry                   4
Sikkim                       4
Andaman & Nicobar Island     3
Daman & Diu                 

In [26]:
print("GeoJSON shape:", geo.shape)

print("\nCRS:")
print(geo.crs)

print("\nColumns:")
print(geo.columns.tolist())

print("\nFirst five records:")
display(geo.head())

GeoJSON shape: (594, 12)

CRS:
EPSG:4326

Columns:
['ID_0', 'ISO', 'NAME_0', 'ID_1', 'NAME_1', 'ID_2', 'NAME_2', 'NL_NAME_2', 'VARNAME_2', 'TYPE_2', 'ENGTYPE_2', 'geometry']

First five records:


,ID_0,ISO,NAME_0,ID_1,NAME_1,ID_2,NAME_2,NL_NAME_2,VARNAME_2,TYPE_2,ENGTYPE_2,geometry
0,105,IND,India,1,Andaman and Nicobar,1,Andaman Islands,None,NaN,District,District,"MULTIPOLYGON (((92.51583 10.89764, 92.51611 10..."
1,105,IND,India,1,Andaman and Nicobar,2,Nicobar Islands,None,NaN,District,District,"MULTIPOLYGON (((93.78773 6.85264, 93.78849 6.8..."
2,105,IND,India,2,Andhra Pradesh,3,Adilabad,None,NaN,District,District,"POLYGON ((78.33625 19.88319, 78.34669 19.8814,..."
3,105,IND,India,2,Andhra Pradesh,4,Anantapur,None,"Anantpur, Ananthapur",District,District,"POLYGON ((77.69 15.17628, 77.69378 15.17347, 7..."
4,105,IND,India,2,Andhra Pradesh,5,Chittoor,None,Chitoor|Chittor,District,District,"POLYGON ((78.47611 13.9368, 78.48208 13.93007,..."


In [28]:
print(geo.columns.tolist())

['ID_0', 'ISO', 'NAME_0', 'ID_1', 'NAME_1', 'ID_2', 'NAME_2', 'NL_NAME_2', 'VARNAME_2', 'TYPE_2', 'ENGTYPE_2', 'geometry']


In [29]:
print("Duplicate district IDs:",
      geo['ID_2'].duplicated().sum())

print("Unique district IDs:",
      geo['ID_2'].nunique())

print("Unique state names:",
      geo['NAME_1'].nunique())

print("Unique district names:",
      geo['NAME_2'].nunique())

Duplicate district IDs: 0
Unique district IDs: 594
Unique state names: 35
Unique district names: 589


In [30]:
print("Districts per state:")

display(
    geo.groupby('NAME_1')['NAME_2']
    .nunique()
    .sort_values(ascending=False)
)

Districts per state:


NAME_1
Uttar Pradesh             70
Madhya Pradesh            48
Bihar                     37
Maharashtra               34
Rajasthan                 32
Tamil Nadu                30
Orissa                    30
Karnataka                 27
Gujarat                   25
Andhra Pradesh            23
Assam                     23
Jharkhand                 22
West Bengal               19
Haryana                   19
Punjab                    17
Chhattisgarh              16
Arunachal Pradesh         15
Jammu and Kashmir         14
Kerala                    14
Uttaranchal               13
Himachal Pradesh          12
Manipur                    9
Mizoram                    8
Nagaland                   8
Meghalaya                  7
Sikkim                     4
Puducherry                 4
Tripura                    4
Andaman and Nicobar        2
Goa                        2
Daman and Diu              2
Lakshadweep                1
Chandigarh                 1
Dadra and Nagar Haveli     1
Delhi  

In [31]:
print("Missing district IDs:", geo['ID_2'].isnull().sum())

print("Missing district names:", geo['NAME_2'].isnull().sum())

print("Missing state names:", geo['NAME_1'].isnull().sum())

Missing district IDs: 0
Missing district names: 0
Missing state names: 0


In [32]:
# Exclude missing district codes
valid_codes = df.dropna(subset=['district_code'])

# Count duplicated state-district code combinations
duplicate_codes = valid_codes[
    valid_codes.duplicated(
        subset=['state_code', 'district_code'],
        keep=False
    )
].sort_values(['state_code', 'district_code'])

print("Records involved in duplicate code combinations:",
      len(duplicate_codes))

display(
    duplicate_codes[
        ['state_name', 'district_name',
         'state_code', 'district_code']
    ]
)

Records involved in duplicate code combinations: 8


,state_name,district_name,state_code,district_code
700,West Bengal,Paschim Medinipur,19,18.0
701,West Bengal,Purba Barddhaman,19,18.0
548,Tamil Nadu,Theni,33,23.0
558,Tamil Nadu,Virudhunagar,33,23.0
570,Telangana,Mahabubabad,36,7.0
571,Telangana,Mahabubnagar,36,7.0
587,Telangana,Warangal Rural,36,9.0
588,Telangana,Warangal Urban,36,9.0


In [33]:
duplicate_pairs = (
    duplicate_codes.groupby(
        ['state_code', 'district_code']
    )
    .size()
    .reset_index(name='record_count')
)

print("Number of duplicated code combinations:",
      len(duplicate_pairs))

display(duplicate_pairs)

Number of duplicated code combinations: 4


,state_code,district_code,record_count
0,19,18.0,2
1,33,23.0,2
2,36,7.0,2
3,36,9.0,2


In [34]:
duplicate_records = df[
    df.duplicated(
        subset=['state_code', 'district_code'],
        keep=False
    ) & df['district_code'].notna()
].sort_values(['state_name', 'district_code'])

columns_to_check = [
    'state_name',
    'district_name',
    'state_code',
    'district_code',
    'stunting',
    'wasting',
    'underweight'
]

display(duplicate_records[columns_to_check])

KeyError: "['stunting', 'wasting', 'underweight'] not in index"

In [35]:
print(df.columns.tolist())

['district_name', 'state_name', 'district_code', 'state_code', 'stunting_pct', 'wasting_pct', 'underweight_pct', 'female_literacy_pct', 'improved_sanitation_pct', 'improved_water_pct', 'anc_4plus_pct', 'child_anemia_pct', 'early_marriage_pct', 'institutional_births_pct']


In [36]:
duplicate_records = df[
    df.duplicated(
        subset=['state_code', 'district_code'],
        keep=False
    ) & df['district_code'].notna()
].sort_values(['state_name', 'district_code'])

columns_to_check = [
    'state_name',
    'district_name',
    'state_code',
    'district_code',
    'stunting_pct',
    'wasting_pct',
    'underweight_pct'
]

display(duplicate_records[columns_to_check])

,state_name,district_name,state_code,district_code,stunting_pct,wasting_pct,underweight_pct
548,Tamil Nadu,Theni,33,23.0,20.2,15.5,18.8
558,Tamil Nadu,Virudhunagar,33,23.0,29.2,14.4,23.7
570,Telangana,Mahabubabad,36,7.0,33.0,19.3,26.3
571,Telangana,Mahabubnagar,36,7.0,42.6,17.8,33.0
587,Telangana,Warangal Rural,36,9.0,36.1,19.0,37.9
588,Telangana,Warangal Urban,36,9.0,29.5,29.5,36.0
700,West Bengal,Paschim Medinipur,19,18.0,31.1,30.3,40.0
701,West Bengal,Purba Barddhaman,19,18.0,32.7,21.1,31.6


display(duplicate_records)

### Finding: Duplicate District Code Combinations

Four combinations of state and district codes were found to be
duplicated, involving eight district records across Tamil Nadu,
Telangana, and West Bengal.

However, the corresponding district names and health indicator
values are different. Therefore, these records represent distinct
district observations and have been retained.

The duplicated codes are flagged for further verification against
the official Census district-code reference. District names combined
with state names are used to distinguish records during the current
data audit.

In [38]:
missing_code_records = df[df['district_code'].isna()]

display(
    missing_code_records[
        ['state_name', 'district_name', 'state_code', 'district_code']
    ].sort_values(['state_name', 'district_name'])
)

,state_name,district_name,state_code,district_code
21,Arunachal Pradesh,Kra Daadi,12,NaN
24,Arunachal Pradesh,Longding,12,NaN
27,Arunachal Pradesh,Namsai,12,NaN
29,Arunachal Pradesh,Siang,12,NaN
38,Assam,Biswanath,18,NaN
...,...,...,...,...
596,Tripura,Unakoti,16,NaN
601,Uttar Pradesh,Amethi,9,NaN
630,Uttar Pradesh,Hapur,9,NaN
661,Uttar Pradesh,Sambhal,9,NaN


### Finding: Missing District Codes

A total of 65 out of 706 records (9.21%) have missing district_code
values.

However, state_name, district_name, and state_code are available
for all 706 records.

The missing district codes may be associated with administrative
changes, newly created districts, or differences in district-code
mapping. This requires verification against an authoritative
district-code reference.

No records have been deleted, and missing district codes have not
been imputed. All district observations are retained for further
analysis.

In [39]:
print("CSV state names:")
print(sorted(df['state_name'].unique()))

print("\nGeoJSON state names:")
print(sorted(geo['NAME_1'].unique()))

CSV state names:
['Andaman & Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra & Nagar Haveli', 'Daman & Diu', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'NCT of Delhi', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

GeoJSON state names:
['Andaman and Nicobar', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Orissa', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Tri

### Finding: GeoJSON and CSV Administrative Compatibility

The factual CSV contains 37 state/UT names, whereas the GeoJSON
contains 35.

Several state/UT names differ due to naming conventions, including
Odisha/Orissa and Uttarakhand/Uttaranchal.

More importantly, Telangana and Ladakh are not represented as
separate administrative units in the current GeoJSON.

The GeoJSON contains 594 district features, compared with 706
district observations in the factual CSV.

Consequently, the current GeoJSON is not suitable for a direct
one-to-one district-level spatial join.

A compatible district boundary dataset must be identified and
validated before spatial analysis.

In [40]:
print("Metadata shape:", metadata.shape)

print("\nMetadata columns:")
print(metadata.columns.tolist())

display(metadata)

Metadata shape: (14, 8)

Metadata columns:
['variable_name', 'indicator_category', 'operational_definition', 'data_type', 'unit', 'nfhs5_indicator_name', 'nfhs5_indicator_number', 'un_sdg_target']


,variable_name,indicator_category,operational_definition,data_type,unit,nfhs5_indicator_name,nfhs5_indicator_number,un_sdg_target
0,district_name,Primary Key / Geographic Identifier,Official administrative name of the district a...,string,District Name,District,NaN,Administrative Geo-Reference
1,state_name,Primary Key / Geographic Hierarchy,Official administrative name of the Indian Sta...,string,State / UT Name,State,NaN,Administrative Geo-Reference
2,district_code,Geospatial Linkage Key,Census of India 2011 district code assigned to...,integer,Census ID,DT_CEN_CD,NaN,Geospatial Boundary Alignment
3,state_code,Geospatial Linkage Key,Census of India 2011 state code assigned to th...,integer,Census ID,ST_CEN_CD,NaN,Geospatial Boundary Alignment
4,stunting_pct,Anthropometric Dependent Variable (Target 2.2),Percentage of children under age 5 years whose...,float,%,Children under 5 years who are stunted (height...,73.0,SDG 2.2.1 (Prevalence of stunting among childr...
5,wasting_pct,Anthropometric Dependent Variable (Target 2.2),Percentage of children under age 5 years whose...,float,%,Children under 5 years who are wasted (weight ...,74.0,SDG 2.2.2 (Prevalence of malnutrition: wasting...
6,underweight_pct,Anthropometric Dependent Variable (Target 2.2),Percentage of children under age 5 years whose...,float,%,Children under 5 years who are underweight (we...,76.0,SDG 2.2 (End all forms of malnutrition by 2030)
7,female_literacy_pct,Multi-Sectoral Predictor / Socioeconomic,Percentage of women age 15-49 years who have c...,float,%,Women who are literate (%),14.0,SDG 4.6 (Adult & Youth Literacy) & SDG 5.b (Wo...
8,improved_sanitation_pct,Multi-Sectoral Predictor / WASH,Percentage of population living in households ...,float,%,Population living in households that use an im...,9.0,SDG 6.2 (Adequate and equitable sanitation and...
9,improved_water_pct,Multi-Sectoral Predictor / WASH,Percentage of population living in households ...,float,%,Population living in households with an improv...,8.0,SDG 6.1 (Safe and affordable drinking water fo...


In [41]:
# Check whether all metadata variables exist in the main dataset

metadata_variables = set(metadata['variable_name'])
dataset_variables = set(df.columns)

print("Variables in metadata:", len(metadata_variables))
print("Variables in dataset:", len(dataset_variables))

print("\nVariables missing from dataset:")
print(sorted(metadata_variables - dataset_variables))

print("\nVariables missing from metadata:")
print(sorted(dataset_variables - metadata_variables))

print("\nDuplicate variable names in metadata:",
      metadata['variable_name'].duplicated().sum())

Variables in metadata: 14
Variables in dataset: 14

Variables missing from dataset:
[]

Variables missing from metadata:
[]

Duplicate variable names in metadata: 0


In [42]:
dtype_comparison = metadata[
    ['variable_name', 'data_type']
].copy()

dtype_comparison['actual_dtype'] = dtype_comparison[
    'variable_name'
].map(df.dtypes.astype(str))

display(dtype_comparison)

,variable_name,data_type,actual_dtype
0,district_name,string,str
1,state_name,string,str
2,district_code,integer,float64
3,state_code,integer,int64
4,stunting_pct,float,float64
5,wasting_pct,float,float64
6,underweight_pct,float,float64
7,female_literacy_pct,float,float64
8,improved_sanitation_pct,float,float64
9,improved_water_pct,float,float64


In [43]:
# Check whether non-missing district codes are whole numbers

valid_codes = df['district_code'].dropna()

non_integer_codes = valid_codes[
    valid_codes % 1 != 0
]

print("Non-missing district codes:", len(valid_codes))
print("Non-integer district codes:", len(non_integer_codes))

Non-missing district codes: 641
Non-integer district codes: 0


In [44]:
# Convert district_code to nullable integer type
df['district_code'] = df['district_code'].astype('Int64')

# Verify
print("District code dtype:", df['district_code'].dtype)
print("Missing codes:", df['district_code'].isna().sum())
print("Non-missing codes:", df['district_code'].notna().sum())

District code dtype: Int64
Missing codes: 65
Non-missing codes: 641


In [46]:
geo_census = gpd.read_file(
    r"../data/geo/Districts.geojson"
)

print("GeoJSON shape:", geo_census.shape)
print("CRS:", geo_census.crs)

GeoJSON shape: (641, 7)
CRS: EPSG:4326


In [47]:
print("Columns:")
print(geo_census.columns.tolist())

print("\nFirst 5 records:")
display(geo_census.head())

print("\nGeometry types:")
print(geo_census.geometry.geom_type.value_counts())

Columns:
['DISTRICT', 'ST_NM', 'ST_CEN_CD', 'DT_CEN_CD', 'censuscode', 'Dist_name', 'geometry']

First 5 records:


,DISTRICT,ST_NM,ST_CEN_CD,DT_CEN_CD,censuscode,Dist_name,geometry
0,Adilabad_1,Telangana,36,1,532.0,Adilabad,"POLYGON ((78.84972 19.7601, 78.85102 19.75945,..."
1,Agra_15,Uttar Pradesh,9,15,146.0,Agra,"POLYGON ((78.19803 27.4028, 78.19804 27.40278,..."
2,Ahmadabad_7,Gujarat,24,7,474.0,Ahmadabad,"MULTIPOLYGON (((72.03456 23.50527, 72.03337 23..."
3,Ahmadnagar_26,Maharashtra,27,26,522.0,Ahmadnagar,"POLYGON ((74.67333 19.9467, 74.67393 19.93509,..."
4,Aizawl_3,Mizoram,15,3,283.0,Aizawl,"POLYGON ((92.98749 24.40453, 92.99107 24.40236..."



Geometry types:
Polygon         567
MultiPolygon     74
Name: count, dtype: int64


In [48]:
csv_states = set(df['state_name'].str.strip().unique())
geo_states = set(geo_census['ST_NM'].str.strip().unique())

print("Unique states in CSV:", len(csv_states))
print("Unique states in GeoJSON:", len(geo_states))

print("\nStates only in CSV:")
print(sorted(csv_states - geo_states))

print("\nStates only in GeoJSON:")
print(sorted(geo_states - csv_states))

Unique states in CSV: 37
Unique states in GeoJSON: 36

States only in CSV:
['Arunachal Pradesh', 'Dadra & Nagar Haveli', 'Ladakh']

States only in GeoJSON:
['Arunanchal Pradesh', 'Dadara & Nagar Havelli']


In [49]:
csv_state_codes = set(df['state_code'].dropna().astype(int))
geo_state_codes = set(geo_census['ST_CEN_CD'].dropna().astype(int))

print("Unique state codes in CSV:", len(csv_state_codes))
print("Unique state codes in GeoJSON:", len(geo_state_codes))

print("\nState codes only in CSV:")
print(sorted(csv_state_codes - geo_state_codes))

print("\nState codes only in GeoJSON:")
print(sorted(geo_state_codes - csv_state_codes))

Unique state codes in CSV: 36
Unique state codes in GeoJSON: 37

State codes only in CSV:
[]

State codes only in GeoJSON:
[99]


In [50]:
extra_state = geo_census[
    geo_census['ST_CEN_CD'] == 99
]

print("Records with state code 99:", len(extra_state))

display(
    extra_state[
        ['DISTRICT', 'ST_NM', 'ST_CEN_CD',
         'DT_CEN_CD', 'censuscode', 'Dist_name']
    ]
)

Records with state code 99: 1


,DISTRICT,ST_NM,ST_CEN_CD,DT_CEN_CD,censuscode,Dist_name
135,Data Not Available_99,Jammu & Kashmir,99,99,0.0,Data Not Available


In [51]:
geo_analysis = geo_census[
    ~(
        (geo_census['ST_CEN_CD'] == 99) &
        (geo_census['DT_CEN_CD'] == 99)
    )
].copy()

print("Original GeoJSON features:", len(geo_census))
print("Working GeoJSON features:", len(geo_analysis))

Original GeoJSON features: 641
Working GeoJSON features: 640


In [52]:
csv_codes = df.dropna(
    subset=['state_code', 'district_code']
).copy()

csv_codes['state_code'] = csv_codes['state_code'].astype('Int64')
csv_codes['district_code'] = csv_codes['district_code'].astype('Int64')

geo_codes = geo_analysis[
    ['ST_CEN_CD', 'DT_CEN_CD', 'ST_NM', 'Dist_name']
].copy()

geo_codes['ST_CEN_CD'] = geo_codes['ST_CEN_CD'].astype('Int64')
geo_codes['DT_CEN_CD'] = geo_codes['DT_CEN_CD'].astype('Int64')

comparison = csv_codes.merge(
    geo_codes,
    left_on=['state_code', 'district_code'],
    right_on=['ST_CEN_CD', 'DT_CEN_CD'],
    how='left',
    indicator=True
)

print("CSV records with non-missing codes:", len(csv_codes))
print("\nMatching results:")
print(comparison['_merge'].value_counts())

CSV records with non-missing codes: 641

Matching results:
_merge
both          641
left_only       0
right_only      0
Name: count, dtype: int64


In [53]:
# Unique Census code combinations in both datasets

csv_unique_codes = csv_codes[
    ['state_code', 'district_code']
].drop_duplicates()

geo_unique_codes = geo_codes[
    ['ST_CEN_CD', 'DT_CEN_CD']
].drop_duplicates()

print("CSV records:", len(csv_codes))
print("Unique CSV code combinations:", len(csv_unique_codes))

print("\nGeoJSON features:", len(geo_analysis))
print("Unique GeoJSON code combinations:", len(geo_unique_codes))

CSV records: 641
Unique CSV code combinations: 637

GeoJSON features: 640
Unique GeoJSON code combinations: 640


In [54]:

code_coverage = csv_unique_codes.merge(
    geo_unique_codes,
    left_on=['state_code', 'district_code'],
    right_on=['ST_CEN_CD', 'DT_CEN_CD'],
    how='outer',
    indicator=True
)

print("Unique code coverage:")
print(code_coverage['_merge'].value_counts())

print("\nCodes present only in CSV:")
display(
    code_coverage[code_coverage['_merge'] == 'left_only']
)

print("\nCodes present only in GeoJSON:")
display(
    code_coverage[code_coverage['_merge'] == 'right_only']
)

Unique code coverage:
_merge
both          637
right_only      3
left_only       0
Name: count, dtype: int64

Codes present only in CSV:


,state_code,district_code,ST_CEN_CD,DT_CEN_CD,_merge



Codes present only in GeoJSON:


,state_code,district_code,ST_CEN_CD,DT_CEN_CD,_merge
298,<NA>,<NA>,17,7,right_only
368,<NA>,<NA>,20,24,right_only
614,<NA>,<NA>,33,24,right_only


In [55]:
# Identify CSV records with duplicated Census code combinations
ambiguous_csv = csv_codes[
    csv_codes.duplicated(
        subset=['state_code', 'district_code'],
        keep=False
    )
].copy()

# Match them to GeoJSON
ambiguous_check = ambiguous_csv.merge(
    geo_codes,
    left_on=['state_code', 'district_code'],
    right_on=['ST_CEN_CD', 'DT_CEN_CD'],
    how='left'
)

display(
    ambiguous_check[
        [
            'state_name',
            'district_name',
            'state_code',
            'district_code',
            'ST_NM',
            'Dist_name'
        ]
    ].sort_values(['state_name', 'district_code', 'district_name'])
)

,state_name,district_name,state_code,district_code,ST_NM,Dist_name
0,Tamil Nadu,Theni,33,23,Tamil Nadu,Theni
1,Tamil Nadu,Virudhunagar,33,23,Tamil Nadu,Theni
2,Telangana,Mahabubabad,36,7,Telangana,Mahbubnagar
3,Telangana,Mahabubnagar,36,7,Telangana,Mahbubnagar
4,Telangana,Warangal Rural,36,9,Telangana,Warangal
5,Telangana,Warangal Urban,36,9,Telangana,Warangal
6,West Bengal,Paschim Medinipur,19,18,West Bengal,Pashchim Medinipur
7,West Bengal,Purba Barddhaman,19,18,West Bengal,Pashchim Medinipur


In [56]:
display(
    extra_geo[
        ['ST_NM', 'Dist_name', 'ST_CEN_CD', 'DT_CEN_CD']
    ]
)

NameError: name 'extra_geo' is not defined

In [57]:
# Find GeoJSON code combinations that do not exist in the CSV

extra_geo = geo_codes.merge(
    csv_unique_codes,
    left_on=['ST_CEN_CD', 'DT_CEN_CD'],
    right_on=['state_code', 'district_code'],
    how='left',
    indicator=True
)

# Keep only GeoJSON-only combinations
extra_geo = extra_geo[
    extra_geo['_merge'] == 'left_only'
].copy()

print("Extra GeoJSON features:", len(extra_geo))

display(
    extra_geo[
        ['ST_NM', 'Dist_name', 'ST_CEN_CD', 'DT_CEN_CD']
    ]
)

Extra GeoJSON features: 3


,ST_NM,Dist_name,ST_CEN_CD,DT_CEN_CD
237,Meghalaya,Jaintia Hills,17,7
501,Jharkhand,Saraikela-kharsawan,20,24
603,Tamil Nadu,Virudunagar,33,24


In [58]:
# Create a normalised name for comparison
csv_names = df[['state_name', 'district_name']].copy()

geo_names = geo_analysis[['ST_NM', 'Dist_name']].copy()

csv_names['state_normalised'] = (
    csv_names['state_name'].str.strip().str.lower()
)

csv_names['district_normalised'] = (
    csv_names['district_name'].str.strip().str.lower()
)

geo_names['state_normalised'] = (
    geo_names['ST_NM'].str.strip().str.lower()
)

geo_names['district_normalised'] = (
    geo_names['Dist_name'].str.strip().str.lower()
)

# Compare state-district name combinations
name_comparison = csv_names.merge(
    geo_names,
    on=['state_normalised', 'district_normalised'],
    how='left',
    indicator=True
)

print("Total CSV records:", len(name_comparison))
print("\nName matching results:")
print(name_comparison['_merge'].value_counts())

Total CSV records: 706

Name matching results:
_merge
both          577
left_only     129
right_only      0
Name: count, dtype: int64


In [59]:

# Extract CSV districts without an exact name match
unmatched_names = name_comparison[
    name_comparison['_merge'] == 'left_only'
].copy()

# Display unique state-district combinations
unmatched_districts = (
    unmatched_names[['state_name', 'district_name']]
    .drop_duplicates()
    .sort_values(['state_name', 'district_name'])
)

print("Unmatched district records:", len(unmatched_names))
print("Unique unmatched state-district pairs:", len(unmatched_districts))

display(unmatched_districts)

Unmatched district records: 129
Unique unmatched state-district pairs: 129


,state_name,district_name
10,Andhra Pradesh,Sri Potti Sriramulu Nellore
15,Andhra Pradesh,Y.S.R.
16,Arunachal Pradesh,Anjaw
17,Arunachal Pradesh,Changlang
18,Arunachal Pradesh,Dibang Valley
...,...,...
698,West Bengal,North Twenty Four Parganas
699,West Bengal,Paschim Barddhaman
700,West Bengal,Paschim Medinipur
701,West Bengal,Purba Barddhaman


In [60]:
unmatched_by_state = (
    unmatched_districts
    .groupby('state_name')
    .size()
    .reset_index(name='unmatched_districts')
    .sort_values('unmatched_districts', ascending=False)
)

print("Number of affected states:", len(unmatched_by_state))
print("Total unmatched districts:",
      unmatched_by_state['unmatched_districts'].sum())

display(unmatched_by_state)

Number of affected states: 25
Total unmatched districts: 129


,state_name,unmatched_districts
20,Telangana,24
1,Arunachal Pradesh,20
6,Gujarat,12
4,Chhattisgarh,11
22,Uttar Pradesh,9
2,Assam,7
13,Meghalaya,6
24,West Bengal,6
18,Sikkim,4
21,Tripura,4


In [61]:
# Inspect unmatched CSV districts and their Census codes

for state in ['Telangana', 'Arunachal Pradesh']:

    print("\n" + "=" * 60)
    print("STATE:", state)
    print("=" * 60)

    print("\nUnmatched CSV districts:")

    display(
        unmatched_names[
            unmatched_names['state_name'] == state
        ][
            ['state_name', 'district_name']
        ].merge(
            df[['state_name', 'district_name',
                'state_code', 'district_code']],
            on=['state_name', 'district_name'],
            how='left'
        ).drop_duplicates()
    )

    geo_state_name = (
        'Arunanchal Pradesh'
        if state == 'Arunachal Pradesh'
        else state
    )

    print("\nDistricts available in GeoJSON:")

    display(
        geo_analysis[
            geo_analysis['ST_NM'] == geo_state_name
        ][
            ['DISTRICT', 'Dist_name',
             'ST_CEN_CD', 'DT_CEN_CD']
        ].sort_values('Dist_name')
    )


STATE: Telangana

Unmatched CSV districts:


,state_name,district_name,state_code,district_code
0,Telangana,Bhadradri Kothagudem,36,<NA>
1,Telangana,Jagitial,36,<NA>
2,Telangana,Jangoan,36,<NA>
3,Telangana,Jayashankar Bhupalapally,36,<NA>
4,Telangana,Jogulamba Gadwal,36,<NA>
5,Telangana,Kamareddy,36,<NA>
6,Telangana,Komaram Bheem Asifabad,36,<NA>
7,Telangana,Mahabubabad,36,7
8,Telangana,Mahabubnagar,36,7
9,Telangana,Mancherial,36,<NA>



Districts available in GeoJSON:


,DISTRICT,Dist_name,ST_CEN_CD,DT_CEN_CD
0,Adilabad_1,Adilabad,36,1
231,Hyderabad_5,Hyderabad,36,5
285,Karimnagar_3,Karimnagar,36,3
296,Khammam_10,Khammam,36,10
352,Mahbubnagar_7,Mahbubnagar,36,7
371,Medak_4,Medak,36,4
396,Nalgonda_8,Nalgonda,36,8
410,Nizamabad_2,Nizamabad,36,2
475,Rangareddy_6,Rangareddy,36,6
607,Warangal_9,Warangal,36,9



STATE: Arunachal Pradesh

Unmatched CSV districts:


,state_name,district_name,state_code,district_code
0,Arunachal Pradesh,Anjaw,12,16
1,Arunachal Pradesh,Changlang,12,9
2,Arunachal Pradesh,Dibang Valley,12,13
3,Arunachal Pradesh,East Kameng,12,3
4,Arunachal Pradesh,East Siang,12,7
5,Arunachal Pradesh,Kra Daadi,12,<NA>
6,Arunachal Pradesh,Kurung Kumey,12,12
7,Arunachal Pradesh,Lohit,12,15
8,Arunachal Pradesh,Longding,12,<NA>
9,Arunachal Pradesh,Lower Dibang Valley,12,14



Districts available in GeoJSON:


,DISTRICT,Dist_name,ST_CEN_CD,DT_CEN_CD
21,Anjaw_16,Anjaw,12,16
108,Changlang_9,Changlang,12,9
154,Dibang Valley_13,Dibang Valley,12,13
169,East Kameng_3,East Kameng,12,3
172,East Siang_7,East Siang,12,7
329,Kurung Kumey_12,Kurung Kumey,12,12
340,Lohit_15,Lohit,12,15
342,Lower Dibang Valley_14,Lower Dibang Valley,12,14
343,Lower Subansiri_11,Lower Subansiri,12,11
430,Papum Pare_4,Papum Pare,12,4


In [62]:
# Create normalised state-district name keys
csv_name_keys = set(
    zip(
        df['state_name'].str.strip().str.lower(),
        df['district_name'].str.strip().str.lower()
    )
)

geo_name_keys = set(
    zip(
        geo_analysis['ST_NM'].str.strip().str.lower(),
        geo_analysis['Dist_name'].str.strip().str.lower()
    )
)

# Create Census code keys
geo_code_keys = set(
    zip(
        geo_analysis['ST_CEN_CD'].astype(int),
        geo_analysis['DT_CEN_CD'].astype(int)
    )
)

# Check exact name matches
df['name_match'] = [
    (state.strip().lower(), district.strip().lower()) in geo_name_keys
    for state, district in zip(df['state_name'], df['district_name'])
]

# Check Census code matches
df['code_match'] = [
    (int(state), int(district)) in geo_code_keys
    if pd.notna(state) and pd.notna(district)
    else False
    for state, district in zip(df['state_code'], df['district_code'])
]

# Classify each record
df['matching_status'] = 'Neither matched'

df.loc[df['name_match'], 'matching_status'] = 'Name matched only'
df.loc[df['code_match'], 'matching_status'] = 'Code matched only'

df.loc[
    df['name_match'] & df['code_match'],
    'matching_status'
] = 'Both name and code matched'

# Display results
print(df['matching_status'].value_counts())

matching_status
Both name and code matched    577
Neither matched                65
Code matched only              64
Name: count, dtype: int64


In [63]:
# Extract records where only the Census code matched
code_only = df[
    df['matching_status'] == 'Code matched only'
].copy()

# Attach the corresponding GeoJSON district name
code_only_details = code_only.merge(
    geo_codes,
    left_on=['state_code', 'district_code'],
    right_on=['ST_CEN_CD', 'DT_CEN_CD'],
    how='left'
)

display(
    code_only_details[
        [
            'state_name',
            'district_name',
            'district_code',
            'ST_NM',
            'Dist_name'
        ]
    ].sort_values(['state_name', 'district_name'])
)

,state_name,district_name,district_code,ST_NM,Dist_name
0,Andhra Pradesh,Sri Potti Sriramulu Nellore,9,Andhra Pradesh,Nellore
1,Andhra Pradesh,Y.S.R.,10,Andhra Pradesh,Cuddapah
2,Arunachal Pradesh,Anjaw,16,Arunanchal Pradesh,Anjaw
3,Arunachal Pradesh,Changlang,9,Arunanchal Pradesh,Changlang
4,Arunachal Pradesh,Dibang Valley,13,Arunanchal Pradesh,Dibang Valley
...,...,...,...,...,...
59,West Bengal,North Twenty Four Parganas,11,West Bengal,North 24 Parganas
60,West Bengal,Paschim Barddhaman,9,West Bengal,Barddhaman
61,West Bengal,Paschim Medinipur,18,West Bengal,Pashchim Medinipur
62,West Bengal,Purba Barddhaman,18,West Bengal,Pashchim Medinipur


In [64]:
print(
    code_only_details[
        [
            'state_name',
            'district_name',
            'district_code',
            'ST_NM',
            'Dist_name'
        ]
    ].sort_values(
        ['state_name', 'district_name']
    ).to_string(index=False)
)

          state_name                district_name  district_code                  ST_NM                    Dist_name
      Andhra Pradesh  Sri Potti Sriramulu Nellore              9         Andhra Pradesh                      Nellore
      Andhra Pradesh                       Y.S.R.             10         Andhra Pradesh                     Cuddapah
   Arunachal Pradesh                        Anjaw             16     Arunanchal Pradesh                        Anjaw
   Arunachal Pradesh                    Changlang              9     Arunanchal Pradesh                    Changlang
   Arunachal Pradesh                Dibang Valley             13     Arunanchal Pradesh                Dibang Valley
   Arunachal Pradesh                  East Kameng              3     Arunanchal Pradesh                  East Kameng
   Arunachal Pradesh                   East Siang              7     Arunanchal Pradesh                   East Siang
   Arunachal Pradesh                 Kurung Kumey             12